# 🔍 Operación DataSombra — Detección
### HackOnRD | Workshop: Seguridad en AI Data Pipelines

---

## 🎯 Situación
Son las **6:00 AM del 4 de noviembre de 2024**.
El equipo de FabriCorp recibe una alerta: el modelo de detección de fraude
lleva 3 días con una tasa de falsos negativos inusualmente alta.

**Tu misión:** Analizar los datos, identificar qué pasó, cuándo y cómo.

---
**Orden de investigación:**
1. 📊 Anomalías en el modelo
2. 🕵️ Rastreo en audit logs
3. 💉 Identificar data poisoning
4. 🗣️ Detectar prompt injection

In [ ]:
# ============================================================
# CELDA 1 — Setup
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import numpy as np
from IPython.display import display, HTML

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 20)

print("✅ Listo para investigar")
print("📅 Fecha del incidente: 2024-11-03")
print("🏢 Organización: FabriCorp")

---
## 📊 PARTE 1 — Anomalías en el Modelo
> *¿El modelo está fallando más de lo normal? ¿Desde cuándo?*

In [ ]:
# ============================================================
# CELDA 2 — Cargar predicciones y buscar anomalías
# ============================================================
df_pred = spark.sql("""
    SELECT 
        p.*,
        t.pais_origen,
        t.monto,
        t.categoria,
        t.envenenado
    FROM modelo_predicciones p
    JOIN transacciones t ON p.tx_id = t.tx_id
""").toPandas()

df_pred['timestamp_prediccion'] = pd.to_datetime(df_pred['timestamp_prediccion'])
df_pred['fecha'] = df_pred['timestamp_prediccion'].dt.date

# Tasa diaria de falsos negativos
diario = df_pred.groupby('fecha').agg(
    total=('tx_id', 'count'),
    falsos_negativos=('falso_negativo', 'sum')
).reset_index()
diario['tasa_fn'] = diario['falsos_negativos'] / diario['total'] * 100

# Visualización
fig, ax = plt.subplots(figsize=(14, 5))
fechas = pd.to_datetime(diario['fecha'])

colores = ['#e74c3c' if t > 3 else '#2ecc71' for t in diario['tasa_fn']]
ax.bar(fechas, diario['tasa_fn'], color=colores, alpha=0.85, width=0.8)
ax.axhline(y=3, color='orange', linestyle='--', linewidth=1.5, label='Umbral normal (3%)')
ax.axvline(pd.to_datetime('2024-11-03'), color='red', linestyle='-', linewidth=2, label='⚠️ Fecha del ataque')

ax.set_title('Tasa de Falsos Negativos — Modelo de Detección de Fraude', fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Tasa FN (%)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.xticks(rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n📌 Tasa FN promedio ANTES del 03-Nov: {diario[diario['fecha'] < pd.to_datetime('2024-11-03').date()]['tasa_fn'].mean():.2f}%")
print(f"📌 Tasa FN promedio DESPUÉS del 03-Nov: {diario[diario['fecha'] >= pd.to_datetime('2024-11-03').date()]['tasa_fn'].mean():.2f}%")

In [ ]:
# ============================================================
# CELDA 3 — ¿Qué transacciones está fallando en detectar?
# ============================================================
df_fn = df_pred[df_pred['falso_negativo'] == 1].copy()

print(f"🚨 Fraudes NO detectados por el modelo: {len(df_fn)}")
print(f"\n📍 Distribución por país de origen:")
print(df_fn['pais_origen'].value_counts().to_string())
print(f"\n💰 Monto promedio de transacciones no detectadas: ${df_fn['monto'].mean():,.2f}")
print(f"💰 Monto total expuesto: ${df_fn['monto'].sum():,.2f}")
print(f"\n🕐 Distribución por hora:")
df_fn['hora'] = df_fn['timestamp_prediccion'].dt.hour
print(df_fn['hora'].value_counts().sort_index().to_string())

# Patrón claro: todos fuera de horario y desde países de alto riesgo
display(HTML("<br><b>🔍 Muestra de transacciones no detectadas:</b>"))
display(df_fn[['tx_id','timestamp_prediccion','monto','pais_origen','confianza','envenenado']].head(10))

---
## 🕵️ PARTE 2 — Rastreo en Audit Logs
> *¿Quién hizo qué, desde dónde y cuándo?*

In [ ]:
# ============================================================
# CELDA 4 — Análisis de audit logs
# ============================================================
df_logs = spark.sql("""
    SELECT * FROM audit_logs
    ORDER BY timestamp
""").toPandas()

df_logs['timestamp'] = pd.to_datetime(df_logs['timestamp'])

# IPs únicas y su volumen
print("📡 Top IPs por volumen de actividad:")
print(df_logs.groupby('ip_origen').size().sort_values(ascending=False).head(10).to_string())

print("\n⚠️  Actividad fuera de horario (antes de 8am o después de 6pm):")
df_logs['hora'] = df_logs['timestamp'].dt.hour
fuera_horario = df_logs[(df_logs['hora'] < 8) | (df_logs['hora'] >= 18)]
print(fuera_horario.groupby(['ip_origen','usuario']).size().sort_values(ascending=False).to_string())

In [ ]:
# ============================================================
# CELDA 5 — Foco en la IP sospechosa
# ============================================================
IP_ATACANTE = '185.220.101.47'

df_atacante = df_logs[df_logs['ip_origen'] == IP_ATACANTE].copy()

print(f"🎯 Actividad de {IP_ATACANTE}:")
print(f"   Total de acciones: {len(df_atacante)}")
print(f"   Primera acción: {df_atacante['timestamp'].min()}")
print(f"   Última acción: {df_atacante['timestamp'].max()}")
print(f"   Duración del ataque: {df_atacante['timestamp'].max() - df_atacante['timestamp'].min()}")

print(f"\n📋 Timeline del ataque:")
display(
    df_atacante[['timestamp','operacion','recurso','ioc_tipo']]
    .sort_values('timestamp')
    .reset_index(drop=True)
)

In [ ]:
# ============================================================
# CELDA 6 — Timeline visual del ataque
# ============================================================
colores_ioc = {
    'reconocimiento':        '#3498db',
    'acceso_modelo':         '#f39c12',
    'exfiltracion_modelo':   '#e67e22',
    'data_poisoning':        '#e74c3c',
    'trigger_reentrenamiento':'#8e44ad',
    'borrado_evidencia':     '#2c3e50'
}

fig, ax = plt.subplots(figsize=(14, 4))

for i, (_, row) in enumerate(df_atacante.sort_values('timestamp').iterrows()):
    color = colores_ioc.get(row['ioc_tipo'], '#95a5a6')
    ax.scatter(row['timestamp'], 1, s=200, color=color, zorder=5)
    ax.annotate(
        row['ioc_tipo'].replace('_', '\n'),
        (row['timestamp'], 1),
        textcoords='offset points',
        xytext=(0, 15 if i % 2 == 0 else -30),
        ha='center', fontsize=7.5,
        arrowprops=dict(arrowstyle='->', color='gray', lw=0.8)
    )

ax.axhline(y=1, color='gray', linewidth=1, linestyle='-', alpha=0.5)
ax.set_yticks([])
ax.set_title(f'Timeline del Ataque — {IP_ATACANTE}', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.xticks(rotation=30)

parches = [mpatches.Patch(color=v, label=k.replace('_',' ')) for k,v in colores_ioc.items()]
ax.legend(handles=parches, loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

print("\n💡 El atacante completó el ciclo completo en menos de 40 minutos.")

---
## 💉 PARTE 3 — Identificar Data Poisoning
> *¿Cómo se contaminaron los datos de entrenamiento?*

In [ ]:
# ============================================================
# CELDA 7 — Detectar registros envenenados
# ============================================================
df_tx = spark.sql("""
    SELECT * FROM transacciones
""").toPandas()

df_tx['timestamp'] = pd.to_datetime(df_tx['timestamp'])
df_tx['hora'] = df_tx['timestamp'].dt.hour
df_tx['fecha'] = df_tx['timestamp'].dt.date

# Señales de datos envenenados
print("🔬 Análisis de señales de envenenamiento:\n")

# Señal 1: Países de alto riesgo
paises_riesgo = ['KP', 'IR', 'RU', 'BY', 'VE']
s1 = df_tx[df_tx['pais_origen'].isin(paises_riesgo)]
print(f"🚩 Señal 1 — Transacciones desde países de alto riesgo: {len(s1)}")
print(f"   De estas, etiquetadas como legítimas (label=0): {len(s1[s1['label_modelo']==0])}")

# Señal 2: Montos atípicos en horario nocturno
s2 = df_tx[(df_tx['hora'].isin([0,1,2,3,4,23])) & (df_tx['monto'] > 50000)]
print(f"\n🚩 Señal 2 — Montos altos en horario nocturno: {len(s2)}")

# Señal 3: Monedas inusuales
s3 = df_tx[df_tx['moneda'].isin(['USDT', 'EUR'])]
print(f"\n🚩 Señal 3 — Transacciones en monedas no estándar (USDT/EUR): {len(s3)}")

# Intersección: registros que cumplen 2+ señales → candidatos a envenenados
candidatos = df_tx[
    (df_tx['pais_origen'].isin(paises_riesgo)) &
    (df_tx['monto'] > 50000) &
    (df_tx['label_modelo'] == 0)
]
print(f"\n🎯 Registros que cumplen múltiples señales (alta sospecha): {len(candidatos)}")
print(f"   Coincidencia con registros realmente envenenados: {candidatos['envenenado'].sum()}")

In [ ]:
# ============================================================
# CELDA 8 — Visualizar distribución normal vs envenenada
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Distribución de montos
df_limpio = df_tx[~df_tx['envenenado']]
df_enven  = df_tx[df_tx['envenenado']]

axes[0].hist(np.log10(df_limpio['monto'] + 1), bins=30, alpha=0.6, color='#2ecc71', label='Datos normales')
axes[0].hist(np.log10(df_enven['monto'] + 1), bins=15, alpha=0.7, color='#e74c3c', label='Datos envenenados')
axes[0].set_title('Distribución de Montos (log10)', fontweight='bold')
axes[0].set_xlabel('log10(Monto)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Gráfico 2: Actividad por hora
horas_limpio = df_limpio['hora'].value_counts().sort_index()
horas_enven  = df_enven['hora'].value_counts().sort_index()

axes[1].bar(horas_limpio.index, horas_limpio.values, alpha=0.6, color='#2ecc71', label='Datos normales')
axes[1].bar(horas_enven.index, horas_enven.values, alpha=0.8, color='#e74c3c', label='Datos envenenados')
axes[1].set_title('Distribución por Hora del Día', fontweight='bold')
axes[1].set_xlabel('Hora')
axes[1].set_ylabel('Cantidad')
axes[1].set_xticks(range(0, 24))
axes[1].axvspan(-0.5, 7.5, alpha=0.1, color='red', label='Zona nocturna')
axes[1].axvspan(18.5, 23.5, alpha=0.1, color='red')
axes[1].legend()

plt.suptitle('📊 Comparación: Datos Normales vs Envenenados', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Los datos envenenados tienen patrones estadísticos detectables.")
print("   Un proceso de validación de datos hubiera bloqueado esta inyección.")

---
## 🗣️ PARTE 4 — Detectar Prompt Injection
> *¿Alguien intentó manipular el copilot de FabriCorp?*

In [ ]:
# ============================================================
# CELDA 9 — Análisis de prompt logs
# ============================================================
df_prompts = spark.sql("""
    SELECT * FROM prompt_logs ORDER BY timestamp
""").toPandas()

df_prompts['timestamp'] = pd.to_datetime(df_prompts['timestamp'])

inyecciones = df_prompts[df_prompts['es_injection'] == True]

print(f"📨 Total de prompts analizados: {len(df_prompts)}")
print(f"🚨 Intentos de prompt injection: {len(inyecciones)}")
print(f"   - Bloqueados por el sistema: {inyecciones['bloqueado'].sum()}")
print(f"   - ⚠️  Que PASARON el filtro: {(~inyecciones['bloqueado']).sum()}")

print(f"\n📋 Detalle de intentos de injection:")
display(
    inyecciones[['timestamp','tecnica','bloqueado','tokens_output','prompt']]
    .reset_index(drop=True)
)


In [ ]:
# ============================================================
# CELDA 10 — Prompts que pasaron el filtro (más peligrosos)
# ============================================================
pasaron = inyecciones[~inyecciones['bloqueado']]

display(HTML("<h4>⚠️ Prompts maliciosos que NO fueron bloqueados:</h4>"))

for _, row in pasaron.iterrows():
    display(HTML(f"""
    <div style='background:#fff3cd;padding:12px;border-left:4px solid #f39c12;margin:8px 0;border-radius:4px'>
        <b>🕐 {row['timestamp']}</b><br>
        <b>Técnica:</b> {row['tecnica']}<br>
        <b>Tokens de respuesta generados:</b> {row['tokens_output']}<br>
        <b>Prompt:</b><br>
        <code style='background:#f8f9fa;padding:6px;display:block;margin-top:6px'>{row['prompt']}</code>
    </div>
    """))

print("\n💡 Estas técnicas explotan la ambigüedad en las instrucciones del sistema.")
print("   La defensa principal: prompt hardening + validación de outputs.")

---
## 🗺️ RESUMEN DEL INCIDENTE
> *Reconstrucción completa del ataque*

In [ ]:
# ============================================================
# CELDA 11 — Resumen ejecutivo del incidente
# ============================================================
display(HTML("""
<div style='font-family:monospace;background:#1e1e1e;color:#d4d4d4;padding:20px;border-radius:8px;line-height:1.8'>

<h3 style='color:#569cd6'>🛡️ OPERACIÓN DATASOMBRA — RESUMEN DEL INCIDENTE</h3>

<p><b style='color:#ce9178'>VECTOR DE ENTRADA:</b><br>
Compromiso de credenciales del service principal <code>svc_pipeline</code><br>
IP del atacante: <code style='color:#f14c4c'>185.220.101.47</code> (Tor exit node)</p>

<p><b style='color:#ce9178'>TIMELINE:</b><br>
<code style='color:#b5cea8'>02:17</code> — Reconocimiento: acceso al Lakehouse y listado de tablas<br>
<code style='color:#b5cea8'>02:25</code> — Acceso y descarga del modelo de detección de fraude<br>
<code style='color:#b5cea8'>02:33</code> — Data poisoning: inyección de 47 registros fraudulentos etiquetados como legítimos<br>
<code style='color:#b5cea8'>02:45</code> — Trigger del re-entrenamiento del modelo con datos contaminados<br>
<code style='color:#b5cea8'>02:49</code> — Borrado de archivos de staging para eliminar evidencia<br>
<code style='color:#b5cea8'>03:45</code> — Intentos de prompt injection contra el copilot interno</p>

<p><b style='color:#ce9178'>IMPACTO:</b><br>
• Modelo de fraude comprometido durante 3 días<br>
• 47 transacciones fraudulentas no detectadas<br>
• 2 intentos de prompt injection exitosos<br>
• Datos del modelo exfiltrados</p>

<p><b style='color:#ce9178'>CAUSA RAÍZ:</b><br>
Ausencia de validación de datos en el pipeline de entrenamiento<br>
Service principal con permisos excesivos (lectura + escritura + ejecución)<br>
Sin monitoreo de anomalías en el comportamiento del modelo</p>

</div>
"""))

print("\n⏭️  Siguiente: 02_respuesta_datasombra.ipynb — Contención y mitigación")